# nb26 — C4（くりこみ/臨界次元）の概念予備調査 ＋ CK-4

**位置づけ（nb24 HOLD 下層・並行調査）.** nb24 で C4「連続極限/臨界次元」は HOLD 下層に置かれた。
着想は：nb19（A-5）の立方異方性 `2|Δφ|² + (4/3)Σφ_c⁴ + …` が**連続回転対称へ流れる臨界次元**として
d=3 が選ばれる、というもの。魅力は**枠内資源（nb19）で安価に試せ密輸が少ない**こと。
HOLD の核心は **「d を出力化できるか」（G1）**だった。

本ノートは**概念予備調査**であり、C4 を本格計算に進めてよいかの**go/kill 判定**だけを下す。
nb24 §5 で**事前登録**した反証チェック **CK-4**（異方性→等方の流れが d=3 でだけ特異/最適になるか。
d=2,3,4,5 を横並びで計算し、3 が際立たなければ後付け確定）を正面から実行する。

D-21 厳守：C4 を**通す**ためでなく、**安価に落とせるなら落とす**ために書く。


## 26.0 C4 の論理構造と、臨界次元論の通常の向き

格子場理論で「臨界次元」は通常 **d を入力**にして相転移のふるまい（連続極限の存在、普遍類）を論じる。
C4 が D-10 を免れる（G1 PASS）には、この向きを**逆転**し、

> 異方性（立方項）が等方（連続回転対称）へ流れること自体が **d を選ぶ**

を示さねばならない。nb19 の展開（A-5）：

```
K(Δφ) = 2|Δφ|²  +  (4/3) Σ_c φ_c⁴  +  (64/45) Σ_c φ_c⁶  + …
        └ 等方(球対称) ┘   └ 立方異方性(格子の軸を見る) ┘
```

最低次の計量 `2|Δφ|²` は**任意の d で等方**（球対称）。破れは立方不変量 `Σ_c φ_c⁴`。
C4 が成立するには、この破れの「等方への回復」が **d=3 で特異/最適**でなければならない。
これを CK-4 で検査する。**ここで d を結論に置かない**（D-10）：d=2..6 を**対称に**並べ、3 が
自然に際立つかだけを見る。


## 26.1 CK-4：立方異方性は d=3 を際立たせるか（d=2..6 横並び）

立方不変量 `Σ_c φ_c⁴` は単位方向 `φ ∈ S^{d-1}` に依存し、`[1/d, 1]` を動く（軸方向で 1、対角で 1/d）。
**異方性の強さ**を、方向平均まわりの **標準偏差**（方向によるばらつき）で測る。
異方性が等方へ「流れて消える」物語が d=3 を選ぶなら、**d=3 で異方性が特異に小さい/大きい**等の
**際立ち**が見えるはず。何も際立たなければ、d=3 は流れの中で特別でない＝後付け（KILL）。


In [1]:

import numpy as np

def cubic_anisotropy(d, samples=400000, seed=0):
    # 単位球 S^{d-1} 上で立方不変量 Σ_c φ_c⁴ の分布統計を返す（std 大=異方性強）。
    rng = np.random.default_rng(seed)
    x = rng.normal(size=(samples, d))
    x /= np.linalg.norm(x, axis=1, keepdims=True)
    quartic = (x**4).sum(1)          # 立方不変量 Σ_c φ_c⁴ ∈ [1/d, 1]
    return quartic.mean(), quartic.std(), quartic.min(), quartic.max()

print("立方異方性 Σ_c φ_c⁴ の方向分布（d 横並び）")
print(f"{'d':>3} {'平均':>8} {'std(異方性)':>12} {'最小=1/d':>10} {'最大':>8}")
prev = None
for d in [2,3,4,5,6]:
    m,s,lo,hi = cubic_anisotropy(d)
    flag = ""
    if prev is not None:
        flag = "↓" if s < prev else "↑"
    print(f"{d:>3} {m:>8.4f} {s:>12.4f} {lo:>10.4f} {hi:>8.4f}   {flag}")
    prev = s


立方異方性 Σ_c φ_c⁴ の方向分布（d 横並び）
  d       平均     std(異方性)     最小=1/d       最大
  2   0.7498       0.1770     0.5000   1.0000   


  3   0.6002       0.1748     0.3333   1.0000   ↓


  4   0.5002       0.1580     0.2500   0.9999   ↓


  5   0.4289       0.1406     0.2009   0.9976   ↓


  6   0.3750       0.1248     0.1678   0.9929   ↓


In [2]:

# CK-4 の判定：std(異方性) が d に対して単調か、d=3 で際立つか
ds = [2,3,4,5,6]
stds = []
for d in ds:
    _, s, _, _ = cubic_anisotropy(d)
    stds.append(s)

# 単調性チェック
diffs = np.diff(stds)
monotone_dec = np.all(diffs < 0)
# d=3 が局所的に際立つ（隣接からの逸脱が大きい）か
mid = stds[1]  # d=3
neighbors_mean = (stds[0] + stds[2]) / 2
standout = abs(mid - neighbors_mean) / neighbors_mean
print(f"std(異方性) の d 系列: {[round(s,4) for s in stds]}")
print(f"単調減少か: {monotone_dec}")
print(f"d=3 の隣接平均からの相対逸脱: {standout:.3%}")
print()
print("="*64)
if monotone_dec and standout < 0.05:
    print("CK-4 判定：FAIL")
    print("  異方性は d とともに *滑らかに単調減少* し、d=3 は際立たない。")
    print("  『等方への流れ』は d=3 を *選ばない*。C4 のこの筋は後付け（D-10 違反の危険）。")
else:
    print("CK-4 判定：要精査（d=3 が際立つ兆候あり）")
print("="*64)


std(異方性) の d 系列: [np.float64(0.177), np.float64(0.1748), np.float64(0.158), np.float64(0.1406), np.float64(0.1248)]
単調減少か: True
d=3 の隣接平均からの相対逸脱: 4.348%

CK-4 判定：FAIL
  異方性は d とともに *滑らかに単調減少* し、d=3 は際立たない。
  『等方への流れ』は d=3 を *選ばない*。C4 のこの筋は後付け（D-10 違反の危険）。


## 26.2 CK-4 の結果の読み（散文）

立方異方性 `Σ_c φ_c⁴` の方向ばらつきは、次元 d とともに **滑らかに単調減少**する。
d=3 は流れの中で**特別な点ではない**（隣接 d=2, d=4 からの際立った逸脱がない）。

これは C4 の素朴版にとって**否定的**である。「異方性が等方へ流れて消えるから d=3」という物語は、
**異方性の消え方が d を選ばない**（高次元ほど一様に等方的になるだけ）以上、成立しない。
nb24 §5 の CK-4 事前登録基準——「d=2,3,4,5 を横並びで計算し、3 が際立たなければ後付け確定」——
に照らし、**この筋は KILL** に近い。

**ただし精密版（D-15）で生き残る余地**を正直に分離しておく：
- 上で測ったのは**幾何（球面上の立方不変量の分布）**であり、**力学（くりこみ群フロー）**ではない。
  くりこみで立方異方性が**関連（relevant）か非関連（irrelevant）か**は d 依存で、
  「立方項が irrelevant になる臨界次元」は別の問いになりうる。
- だがそれは **d を入力にした演算子次元の解析**＝臨界次元論の**通常の向き**で、まさに nb24 が
  G1（d を出力化できるか）で警告した循環の入口。逆転できる保証はない。


## 26.3 判定：C4 は素朴版 KILL、精密版は「高コスト HOLD」へ降格

肯定・否定を同じ精度で（D-18）：

- **素朴版（幾何的異方性が d=3 を選ぶ）：KILL.** CK-4 で否定。異方性は d で滑らかに単調減少し
  d=3 を際立たせない。nb24 が期待した「枠内資源で安価に」は**安価に否定**できた——これは成功
  （空振りも否定的確定として価値、D-15/D-24）。
- **精密版（くりこみで立方項が irrelevant になる臨界次元）：高コスト HOLD.** 成立余地はあるが、
  (i) 力学（くりこみ群フロー）の導入が要り、(ii) d を入力にする通常の向きで、G1 循環の罠が濃い。
  C8（nb25）の「絡みの実現を与件から言う」道に比べ、**D-10 免除の見通しが悪い**。
  → C4 は当面 **C8 の後回し**。nb25 で C8 が GO へ進めば C4 の精密版は「独立証拠の合流」(nb25 §25.5 道ii)
  としてのみ価値を持つ。

**nb24 への反映**：C4 は HOLD 下層 → 素朴版 KILL ＋ 精密版・高コスト HOLD に**精密化**。
優先順位は C8（nb25）が単独首位で確定。


In [3]:

print('''
nb24 候補表への更新（nb25・nb26 の結果反映）
================================================
  C8 絡みの非自明性 : HOLD上層 → 「記述は強・選択は未完」へ精密化（nb25）
                       残課題は『絡みの実現を与件から言う』一点。最優先で継続。
  C4 臨界次元       : HOLD下層 → 素朴版 KILL（CK-4, nb26）＋ 精密版・高コスト HOLD
                       C8 が GO へ進んだ場合のみ『独立証拠の合流』として再浮上。

次の一手の確定：nb27 = C8 の残課題（non-signaling/因果律から相関の位相的非自明性を
                要求できるか）。これが閉じれば C8 は GO、次元3の上流原理の初の候補確立。
''')



nb24 候補表への更新（nb25・nb26 の結果反映）
  C8 絡みの非自明性 : HOLD上層 → 「記述は強・選択は未完」へ精密化（nb25）
                       残課題は『絡みの実現を与件から言う』一点。最優先で継続。
  C4 臨界次元       : HOLD下層 → 素朴版 KILL（CK-4, nb26）＋ 精密版・高コスト HOLD
                       C8 が GO へ進んだ場合のみ『独立証拠の合流』として再浮上。

次の一手の確定：nb27 = C8 の残課題（non-signaling/因果律から相関の位相的非自明性を
                要求できるか）。これが閉じれば C8 は GO、次元3の上流原理の初の候補確立。



## 26.4 サマリー（established / open）

### established（本ノートで確定）
1. **立方異方性 `Σ_c φ_c⁴` の方向ばらつきは d とともに滑らかに単調減少し、d=3 を際立たせない**（CK-4）。
2. ゆえに C4 の**素朴版**（幾何的異方性の等方化が d=3 を選ぶ）は **KILL**。安価な否定的確定（D-15/D-24）。
3. C4 の**精密版**（くりこみで立方項が irrelevant になる臨界次元）は成立余地ありだが、力学導入＋
   d を入力にする通常の向きで **G1 循環の罠が濃い高コスト HOLD**。C8 の後回し。

### open（次へ）
1. **C8 の残課題（nb27 で着手）.** non-signaling／因果律から「相関の位相的非自明性（絡みの実現）」を
   要求できるか。閉じれば C8 は GO。
2. C4 精密版は、C8 が GO に達した場合のみ「独立証拠の合流」として再評価する（nb25 §25.5 道ii）。

> **D-10 厳守の確認.** 本ノートは d=2..6 を**対称に**並べ、d=3 を結論に置かずに「際立つか」だけを問うた。
> 結果は「際立たない」＝C4 素朴版の否定であり、次元3を後付けする誘惑を**自ら退けた**記録である。
